# Phase 1 — Dataset Preparation and Inspection
### Problem Statement ID 26184: Predictive Analytics Framework for Cybercrime Complaints

This notebook explores the six raw cybercrime and ATM withdrawal datasets, validating data quality, geospatial coordinates, and temporal distributions.


## 1. Catalog and Load Raw Datasets


In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
datasets = {
    'Fraud_Cases': pd.read_csv(RAW_DIR / 'Fraud_Cases.csv'),
    'Withdrawals': pd.read_csv(RAW_DIR / 'Withdrawals.csv'),
    'Transactions': pd.read_csv(RAW_DIR / 'Transactions.csv'),
    'ATMs_Locations': pd.read_csv(RAW_DIR / 'ATMs_Locations.csv'),
    'Areas_Master': pd.read_csv(RAW_DIR / 'Areas_Master.csv'),
    'Accounts': pd.read_csv(RAW_DIR / 'Accounts.csv')
}
for name, df in datasets.items():
    print(f'{name:16s}: {df.shape[0]:,} rows, {df.shape[1]} columns')


## 2. Inspect Cybercrime Complaints (`Fraud_Cases`)


In [ ]:
fraud = datasets['Fraud_Cases']
print('Fraud Cases Head:')
display(fraud.head())
print('\nCrime Type Breakdown:')
print(fraud['crime_type'].value_counts())


## 3. Inspect Ground Truth Cashouts (`Withdrawals`)


In [ ]:
withd = datasets['Withdrawals']
print('Withdrawals Head:')
display(withd.head())
print(f'Fraud-Linked Withdrawals: {withd["case_id"].notna().sum():,}')
print(f'Background Withdrawals: {withd["case_id"].isna().sum():,}')


## 4. Geospatial Verification


In [ ]:
print('Withdrawals Coordinate Bounds:')
print('Latitude :', withd['latitude'].min(), 'to', withd['latitude'].max())
print('Longitude:', withd['longitude'].min(), 'to', withd['longitude'].max())


## 5. Complaint-to-Cashout Delay Analysis


In [ ]:
fraud['complaint_dt'] = pd.to_datetime(fraud['complaint_timestamp'])
withd['withd_dt'] = pd.to_datetime(withd['timestamp'])
merged = pd.merge(withd.dropna(subset=['case_id']), fraud, on='case_id')
delays = (merged['withd_dt'] - merged['complaint_dt']).dt.total_seconds() / 3600.0
print('Complaint-to-Cashout Delay (Hours):')
print(delays.describe())
